# Trajectory Dataset Generator
Executing paths explicitly via dynamic `InfRRTStar` models bounded across generic arrays natively written into standalone Minari `./data/` directories.

In [ ]:
# PARAMETERS
# Select environment: 'EnvUMazeRandomEllipsoids2D' or 'EnvLargeMazeRandomEllipsoids2D'
ENVIRONMENT_NAME = 'EnvLargeMazeRandomEllipsoids2D' 
TARGET_EPISODES = 5000
DATASET_ID = "custom/largemaze_ellipsoids-v1"
RRT_MAX_TIME = 8.0 # Max time to spend finding a path per episode
MAX_OBSTACLES = 5 # Dynamic padding constraint bounds
DIST_FACTOR = 0.25 # Minimum start-to-goal diagonal margin bounds

# Where datasets are stored — set this explicitly so you always know where your data is.
# Default: ~/.minari/datasets (system store, persistent across sessions)
# Uncomment to use a local folder instead:
# DATA_DIR = "/home/earth/SDPMSP/Diffuser/TrajectoryDatasetGeneration/datasets"
DATA_DIR = "/home/earth/SDPMSP/Diffuser/TrajectoryDatasetGeneration/data"  # None = use the default system Minari store (~/.minari/datasets)


In [33]:
import os
import sys
import torch
import numpy as np
from gymnasium.spaces import Box
import minari
from minari import EpisodeData

# Mapping backend paths natively
sys.path.append('/home/earth/MPD/mpd-public')
sys.path.append('/home/earth/MPD/mpd-public/deps/torch_robotics')

from trajectory_generator import EnvUMazeRandomEllipsoids2D, EnvLargeMazeRandomEllipsoids2D, generate_trajectory
from torch_robotics.robots import RobotPointMass
from torch_robotics.tasks.tasks import PlanningTask

tensor_args = {'device': 'cpu', 'dtype': torch.float32}

# Configure data storage path
if DATA_DIR is not None:
    os.makedirs(DATA_DIR, exist_ok=True)
    os.environ['MINARI_DATASETS_PATH'] = os.path.abspath(DATA_DIR)
    print(f"Storing datasets in: {os.path.abspath(DATA_DIR)}")
else:
    print(f"Storing datasets in system Minari store (~/.minari/datasets)")


Storing datasets in: /home/earth/SDPMSP/Diffuser/TrajectoryDatasetGeneration/data


In [34]:
# Dataset Generation Loop
print(f"===========================================")
print(f"Beginning dataset generation for: {DATASET_ID}")

env_class = EnvLargeMazeRandomEllipsoids2D if ENVIRONMENT_NAME == 'EnvLargeMazeRandomEllipsoids2D' else EnvUMazeRandomEllipsoids2D

ep = 0
episodes_buffer = []

while ep < TARGET_EPISODES:
    print(f"--- Episode {ep+1} ---")
    env = env_class(tensor_args=tensor_args, max_obstacles=MAX_OBSTACLES)
    robot = RobotPointMass(q_limits=env.limits, tensor_args=tensor_args)
    task = PlanningTask(env=env, robot=robot, obstacle_cutoff_margin=0.08, tensor_args=tensor_args)

    min_dist = torch.norm(env.limits[1] - env.limits[0]) * DIST_FACTOR 
    
    found_start_goal = False
    for _ in range(500):
        q_free = task.random_coll_free_q(n_samples=2)
        start_p, goal_p = q_free[0], q_free[1]
        if torch.norm(start_p - goal_p) > min_dist:
            start_pos, goal_pos = start_p, goal_p
            found_start_goal = True
            break
    
    if not found_start_goal:
        continue
        
    success, free_trajs = generate_trajectory(env, robot, task, start_pos, goal_pos, tensor_args, gpmp_opt_iters=200, rrt_max_time=RRT_MAX_TIME)
    
    if success:
        print(f"  - Generating EpisodeData ({ep+1}/{TARGET_EPISODES})")
        traj = free_trajs[0].detach().cpu().numpy()
        actions = traj[:, 2:4][:-1] if traj.shape[-1] == 4 else traj[1:] - traj[:-1]
        observations = traj
        rewards = np.zeros(len(actions))
        terminations = np.array([False]*(len(actions)-1) + [True])
        truncations = np.array([False]*len(actions))

        padded_centers = np.zeros((MAX_OBSTACLES, 2), dtype=np.float32)
        padded_centers[:len(env.ellipsoids_centers)] = env.ellipsoids_centers
        padded_radii = np.zeros((MAX_OBSTACLES, 2), dtype=np.float32)
        padded_radii[:len(env.ellipsoids_radii)] = env.ellipsoids_radii

        centers_tile = np.tile(padded_centers, (len(actions), 1, 1))
        radii_tile = np.tile(padded_radii, (len(actions), 1, 1))
        start_tile = np.tile(start_pos.cpu().numpy().astype(np.float32), (len(actions), 1))
        goal_tile = np.tile(goal_pos.cpu().numpy().astype(np.float32), (len(actions), 1))

        episode_info = {
            "ellipsoids_centers": centers_tile,
            "ellipsoids_radii": radii_tile,
            "start_pos": start_tile,
            "goal_pos": goal_tile,
        }

        episode_data = EpisodeData(
            id=ep,
            observations=observations.astype(np.float32),
            actions=actions.astype(np.float32),
            rewards=rewards.astype(np.float32),
            terminations=terminations,
            truncations=truncations,
            infos=episode_info
        )
        object.__setattr__(episode_data, 'seed', None)
        object.__setattr__(episode_data, 'options', None)
        episodes_buffer.append(episode_data)
        ep += 1

print(f"Saving compiled Minari Dataset: {DATASET_ID} to ./data")
obs_space = Box(low=-np.inf, high=np.inf, shape=(observations.shape[-1],), dtype=np.float32)
act_space = Box(low=-np.inf, high=np.inf, shape=(actions.shape[-1],), dtype=np.float32)

# Append to existing dataset, or create a new one if it doesn't exist
# Robust existence check — list_local_datasets crashes if any entry has a malformed ID
def dataset_exists(dataset_id):
    try:
        return dataset_id in minari.list_local_datasets()
    except (TypeError, ValueError):
        # Fallback: check the HDF5 file directly
        import pathlib
        data_path = pathlib.Path(os.environ['MINARI_DATASETS_PATH']) / dataset_id / 'data' / 'main_data.hdf5'
        return data_path.exists()

if dataset_exists(DATASET_ID):
    print(f"  Dataset already exists — appending {TARGET_EPISODES} new episodes...")
    existing = minari.load_dataset(DATASET_ID)
    # total_episodes can be None if metadata wasn't fully flushed — count from storage directly
    episodes_before = existing.total_episodes
    if episodes_before is None:
        import h5py
        with h5py.File(existing._data._file_path, 'r') as f:
            episodes_before = len(f.keys())
    episodes_before = int(episodes_before)
    # Re-ID episodes sequentially from the existing count so they don't overwrite
    for i, ep_data in enumerate(episodes_buffer):
        object.__setattr__(ep_data, 'id', episodes_before + i)
    existing._data.update_episodes(episodes_buffer)
    dataset = minari.load_dataset(DATASET_ID)
    print(f"  Episodes before: {episodes_before} → after: {dataset.total_episodes}")
else:
    print(f"  Creating new dataset...")
    dataset = minari.create_dataset_from_buffers(
        dataset_id=DATASET_ID,
        buffer=episodes_buffer,
        observation_space=obs_space,
        action_space=act_space,
        algorithm_name="InfRRTStar_GPMP2",
        author="MPD Parametric Pipeline"
    )
    print(f"  New dataset created with {dataset.total_episodes} episodes.")


Beginning dataset generation for: custom/largemaze_ellipsoids-v1
--- Episode 1 ---
Precomputing the SDF grid and gradients took: 0.605 sec
Iteration:  6126/25000 | Time: 8.000 s| Nodes: 6128 | Success: False | Cost: inf
--- Episode 1 ---
Precomputing the SDF grid and gradients took: 0.553 sec
Iteration:  2112/25000 | Time: 1.283 s| Nodes: 1010 | Success: True | Cost: 7.194168090820
  - Generating EpisodeData (1/5)
--- Episode 2 ---
Precomputing the SDF grid and gradients took: 0.504 sec
Iteration:  1480/25000 | Time: 0.752 s| Nodes: 975 | Success: True | Cost: 9.017838478088
  - Generating EpisodeData (2/5)
--- Episode 3 ---
Precomputing the SDF grid and gradients took: 0.588 sec
Iteration: 10432/25000 | Time: 8.001 s| Nodes: 4842 | Success: True | Cost: 7.292541503906
  - Generating EpisodeData (3/5)
--- Episode 4 ---
Precomputing the SDF grid and gradients took: 0.646 sec
Iteration:  3414/25000 | Time: 2.915 s| Nodes: 3287 | Success: True | Cost: 13.517798423767
  - Generating Episod